# Evaluation audit (September 2026) — part A
### Contamination check · deduplicated held-out set · re-scoring · tokenizer ablation

This notebook re-measures every bits-per-byte figure for **uzbek-gpt-103m** after an audit found that the original held-out set was the opening of the training split.

| Step | What it does | Script |
|---|---|---|
| 1 | Shows the original held-out set is inside `train.bin` (11 of 12 probes) | `01_contamination_check.py` |
| 2 | Builds a deduplicated held-out set, proves it is clean, fine-tunes the QLoRA baseline, scores every model two ways, runs the paired bootstrap | `02_clean_eval_and_score.py` |
| 3 | Scores the 232M tokenizer-ablation model on the clean set | `03_score_ablation.py` |

**To run on Kaggle:** GPU T4, internet on, and add two inputs: the `uzbek-fineweb2-tokens-16k` dataset (`train.bin`, `val.bin`) and the `ablation-best` dataset (`ablation_best.pt`). About 60–90 minutes.

**About the outputs.** They are this session's own outputs, with download progress bars and library loading warnings removed. The same text is saved in `logs/`.

**About the numbers.** This is the second of three sessions run with identical code. Every deterministic measurement (from-scratch, zero-shot mGPT, ablation) is identical across all three. The QLoRA baseline involves GPU training, which is not bit-for-bit deterministic: this session gives 1.0766 bits/byte, while the first session, whose figures are published in the paper and in `results/results_v3.json`, gave 1.0764. See the README for all runs.

In [ ]:
!pip install -q transformers datasets peft bitsandbytes accelerate safetensors huggingface_hub

## Step 1 — Is the original held-out set really unseen?

The original held-out set was built by streaming FineWeb-2 from its beginning. The corpus had been split 90/10 **by position** into `train.bin` and `val.bin`, so the beginning of the stream is training data.

The check takes 12 passages of 32 tokens from across the held-out text and searches for the exact token sequence in both files. A 32-token exact match cannot happen by chance.

Note where the matches land: eval position 0 → train position 0, 35,157 → 35,157, 210,944 → 210,952. The small drift is the `<|endoftext|>` separators added between documents. The held-out set is not overlapping the training data; it **is** the opening of it.

In [ ]:
"""
================================================================================
CONTAMINATION CHECK — is the eval set inside the from-scratch model's TRAINING data?
================================================================================
Runs in ~2 minutes. No GPU needed. Run this before 02_clean_eval_and_score.py.

THE SUSPICION
  tokenize_data.py splits FineWeb-2 uzn_Latn 90/10 into train.bin / val.bin.
  A positional 90/10 split means train.bin = the FIRST 90% of the document stream.
  uz_heldout.txt was built by streaming FineWeb-2 from the START and taking the
  first ~200k words. If both are true, the "held-out" eval set sits inside
  train.bin, and uzbek-gpt-103m was trained on its own test set for ~2 epochs.

  That would make 1.105 bpb too low, in the direction that favours the paper's
  headline. It has to be checked before anything else is worth running.

WHAT IT DOES
  Takes several probe passages from the eval text, tokenizes them with
  uzbek-bpe-16k, and searches train.bin and val.bin for the exact token sequence.
  Exact-match search on token ids — no false positives.

ATTACH THE DATASET FIRST
  Notebook sidebar -> Add Input -> your dataset `uzbek-fineweb2-tokens-16k`.
  It mounts at /kaggle/input/uzbek-fineweb2-tokens-16k/
================================================================================
"""

import os, glob
import numpy as np
from transformers import AutoTokenizer
from datasets import load_dataset

TOK_REPO   = "IslombekT/uzbek-bpe-16k"
EVAL_FILE  = "/kaggle/working/uz_heldout.txt"
N_PROBES   = 12          # passages to test
PROBE_TOK  = 32          # tokens per probe; 32 is far beyond chance
EVAL_WORDS = 200_000

# ---- locate the bins -------------------------------------------------------
cands = glob.glob("/kaggle/input/**/train.bin", recursive=True)
if not cands:
    raise SystemExit("train.bin not found. Add Input -> uzbek-fineweb2-tokens-16k")
TRAIN_BIN = cands[0]
VAL_BIN   = os.path.join(os.path.dirname(TRAIN_BIN), "val.bin")
print(f"train.bin: {TRAIN_BIN}")
print(f"val.bin  : {VAL_BIN} ({'found' if os.path.exists(VAL_BIN) else 'MISSING'})")

train = np.memmap(TRAIN_BIN, dtype=np.uint16, mode="r")
val   = np.memmap(VAL_BIN, dtype=np.uint16, mode="r") if os.path.exists(VAL_BIN) else None
print(f"train tokens: {len(train):,}   val tokens: {len(val):,}" if val is not None
      else f"train tokens: {len(train):,}")

# ---- get the eval text -----------------------------------------------------
if os.path.exists(EVAL_FILE):
    TEXT = open(EVAL_FILE, encoding="utf-8").read()
    print(f"using existing {EVAL_FILE}")
else:
    print("rebuilding the eval text from the FineWeb-2 stream (same recipe as before)...")
    ds = load_dataset("HuggingFaceFW/fineweb-2", name="uzn_Latn",
                      split="train", streaming=True)
    docs, w = [], 0
    for ex in ds:
        t = (ex.get("text") or "").strip()
        if not t:
            continue
        docs.append(t); w += len(t.split())
        if w >= EVAL_WORDS:
            break
    TEXT = "\n".join(docs)
print(f"eval text: {len(TEXT):,} chars, {len(TEXT.split()):,} words")

tok = AutoTokenizer.from_pretrained(TOK_REPO)
tok.model_max_length = 10**9
eval_ids = np.asarray(tok(TEXT, add_special_tokens=False)["input_ids"], dtype=np.uint16)
print(f"eval text -> {len(eval_ids):,} tokens under uzbek-bpe-16k\n")


def find_sequence(hay, needle):
    """Exact search for a token sequence. Vectorized progressive filter."""
    L = len(needle)
    if L == 0 or len(hay) < L:
        return []
    cand = np.flatnonzero(hay[:len(hay) - L + 1] == needle[0])
    for k in range(1, L):
        if cand.size == 0:
            break
        cand = cand[hay[cand + k] == needle[k]]
    return cand.tolist()


# ---- probe -----------------------------------------------------------------
positions = np.linspace(0, len(eval_ids) - PROBE_TOK - 1, N_PROBES).astype(int)
hits_train, hits_val = 0, 0

print(f"{'probe':>6}  {'eval token pos':>14}  {'in train.bin':>13}  {'in val.bin':>11}")
print("-" * 52)
for n, p in enumerate(positions, 1):
    needle = np.asarray(eval_ids[p:p + PROBE_TOK], dtype=np.uint16)
    t_hit = find_sequence(train, needle)
    v_hit = find_sequence(val, needle) if val is not None else []
    hits_train += bool(t_hit)
    hits_val += bool(v_hit)
    print(f"{n:>6}  {p:>14,}  {('YES @'+format(t_hit[0],',')) if t_hit else 'no':>13}  "
          f"{('YES @'+format(v_hit[0],',')) if v_hit else 'no':>11}")

print("-" * 52)
print(f"\n{hits_train}/{N_PROBES} probes found in train.bin")
print(f"{hits_val}/{N_PROBES} probes found in val.bin")

print()
if hits_train > 0:
    print("=" * 72)
    print("CONTAMINATED. The eval set is inside the from-scratch model's training")
    print("data. uzbek-gpt-103m saw this text for ~2 epochs, so 1.105 bpb is too")
    print("low, in the direction that flatters the paper's headline. Every bpb")
    print("number measured on this eval set is affected, including the ablation.")
    print("Do not evaluate against this eval set.")
    print("=" * 72)
elif hits_val > 0:
    print("Eval text sits in val.bin — genuinely held out from the from-scratch")
    print("model. The published numbers stand on this point.")
else:
    print("Not found in either bin. Either the probes crossed a document boundary")
    print("that got an <|endoftext|> inserted, or the eval text came from a")
    print("different stream position than these bins. Report this output as-is.")


train.bin: /kaggle/input/datasets/islombekturdiyev/uzbek-fineweb2-tokens-16k/train.bin
val.bin  : /kaggle/input/datasets/islombekturdiyev/uzbek-fineweb2-tokens-16k/val.bin (found)
train tokens: 954,981,404   val tokens: 106,109,045
rebuilding the eval text from the FineWeb-2 stream (same recipe as before)...
eval text: 1,645,017 chars, 203,257 words
eval text -> 386,765 tokens under uzbek-bpe-16k

 probe  eval token pos   in train.bin   in val.bin
----------------------------------------------------
     1               0         YES @0           no
     2          35,157    YES @35,157           no
     3          70,314    YES @70,320           no
     4         105,472   YES @105,478           no
     5         140,629   YES @140,635           no
     6         175,787   YES @175,793           no
     7         210,944   YES @210,952  YES @13,027,100
     8         246,102   YES @246,116           no
     9         281,259   YES @281,273           no
    10         316,417   YES @31

## Step 2 — Build a clean held-out set and re-measure everything

A first attempt drew the held-out set straight from `val.bin`. It was rejected: 6 of 16 passages still appeared in `train.bin`, at scattered positions, because FineWeb-2 contains duplicate web pages.

So this step filters **document by document**:

1. Fingerprint every 24-token window of `train.bin` (27.4 million fingerprints).
2. Keep only `val.bin` documents with **zero** fingerprint overlap: 474 kept, 306 rejected (39.2%).
3. Verify with exact search, independently of the fingerprints. The script stops if any passage is found.

Then it fine-tunes the mGPT-1.3B + QLoRA baseline on 1M tokens (checked to share nothing with the held-out text) and scores every model under two protocols:

- **Protocol A** — 512-token chunks per model, the original design. It gives the more efficient tokenizer more text per chunk, and chunks do not line up across models.
- **Protocol B** — the text is cut into ~800-byte spans first, so every model scores identical text with identical context. This is the headline protocol, and it is what makes a **paired** bootstrap possible.

In [ ]:
"""
================================================================================
UZBEK-GPT PAPER — VERIFICATION RUN v3  (DEDUPLICATED EVAL SET)
================================================================================
Why v3 exists:

  v1  eval set was the literal opening of train.bin. Void.
  v2  eval set decoded from val.bin — but 6/16 probes still turned up in
      train.bin, at scattered positions (252M, 929M, 833M, ...). That is not a
      split error. FineWeb-2 itself contains the same passages in both halves.
      So val.bin is partly contaminated too, and no choice of slice fixes it.
  v3  filters DOCUMENT BY DOCUMENT. Builds a fingerprint index of every 24-token
      window in train.bin, then keeps only val documents with zero fingerprint
      overlap. What survives is text the from-scratch model provably never saw.

CONSEQUENCE YOU SHOULD ABSORB
  Your model's reported validation loss of 3.059 was measured on val.bin, which
  we now know is partly duplicated from train.bin. That number is optimistic too.
  This is a property of the corpus, not a mistake you made — but the paper has to
  say the eval set was deduplicated against the training split, because from now
  on it will have been.

STAGES
  1  fingerprint index of train.bin            (~3 min, one pass)
  2  walk val.bin, drop duplicated documents, build the clean eval set
  3  verify with exact search — must be 0/16
  4  rebuild the old contaminated set, for the memorisation delta
  5  adaptation pool + text-level leak check
  6  QLoRA fine-tune mGPT-1.3B, ctx 512, all-linear, ~1M tokens
  7  Protocol A (512-token chunks): 3 models on clean + from-scratch on old
  8  Protocol B (byte-aligned spans): 3 models on clean
  9  paired bootstrap
  10 RESULTS

RUNTIME ~70-90 min on a T4. Restartable; every stage caches to /kaggle/working.

FIRST, IN A SEPARATE CELL:
    !pip install -q transformers datasets peft bitsandbytes accelerate safetensors huggingface_hub
Accelerator: GPU T4. Internet: ON. Add Input: uzbek-fineweb2-tokens-16k.
================================================================================
"""

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import gc, sys, math, json, csv, glob, time
import numpy as np
import torch

LN2  = math.log(2.0)
WORK = "/kaggle/working"
os.makedirs(WORK, exist_ok=True)

# ------------------------------- config -------------------------------------
BASE          = "ai-forever/mGPT"
MY_MODEL_REPO = "IslombekT/uzbek-gpt-103m"
MY_TOK_REPO   = "IslombekT/uzbek-bpe-16k"
ADAPTER_DIR   = f"{WORK}/mgpt-uz-qlora-ctx512-dedup"

EOT_ID        = 0
KGRAM         = 24          # fingerprint window, ~18 Uzbek words
KEEP_MASK     = 31          # keep 1/32 of windows in the index
MAX_DUP_FRAC  = 0.0         # a document is rejected on ANY fingerprint match
MIN_DOC_WORDS = 30          # skip stubs, too short to fingerprint meaningfully
EVAL_WORDS    = 200_000

ADAPT_TOKENS  = 1_000_000
CTX_TRAIN     = 512
PER_DEV, ACC  = 2, 2
LR            = 2e-4

CHUNK_TOKENS  = 512
SPAN_BYTES    = 800
CONTEXT_BYTES = 300
MAX_TOKENS    = 512

N_PROBES      = 16
PROBE_TOK     = 32
RESAMPLES     = 10_000
SEED          = 20260913

assert torch.cuda.is_available(), "Enable a GPU: Settings -> Accelerator -> GPU T4."
DEV  = "cuda"
BF16 = torch.cuda.get_device_capability(0)[0] >= 8
DT   = torch.bfloat16 if BF16 else torch.float16
print(f"GPU: {torch.cuda.get_device_name(0)} | bf16: {BF16}")

def free(): gc.collect(); torch.cuda.empty_cache()
def stage(n, name): print(f"\n{'='*72}\nSTAGE {n} — {name}\n{'='*72}")

R = {"config": {"kgram": KGRAM, "eval_words": EVAL_WORDS, "adapt_tokens": ADAPT_TOKENS,
                "ctx_train": CTX_TRAIN, "span_bytes": SPAN_BYTES,
                "resamples": RESAMPLES, "seed": SEED}}

from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
                          TrainingArguments, Trainer, default_data_collator)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
from torch.utils.data import Dataset

_B = np.uint64(1000003)

def kgram_hashes(arr, K=KGRAM, mask=KEEP_MASK, chunk=20_000_000):
    """Rolling polynomial hash of every K-gram, subsampled to 1/(mask+1)."""
    out, n = [], len(arr)
    if n < K:
        return np.zeros(0, dtype=np.uint64)
    for s in range(0, n - K + 1, chunk):
        e = min(s + chunk, n - K + 1); m = e - s
        h = np.zeros(m, dtype=np.uint64)
        for j in range(K):
            h = h * _B + arr[s+j : s+j+m].astype(np.uint64)
        out.append(h[(h & np.uint64(mask)) == 0])
    return np.concatenate(out) if out else np.zeros(0, dtype=np.uint64)

def find_sequence(hay, needle):
    L = len(needle)
    if L == 0 or len(hay) < L: return []
    cand = np.flatnonzero(hay[:len(hay)-L+1] == needle[0])
    for k in range(1, L):
        if cand.size == 0: break
        cand = cand[hay[cand+k] == needle[k]]
    return cand.tolist()

# =============== STAGE 1 — fingerprint index of train.bin ===================
stage(1, "fingerprint every 24-token window of train.bin")
cands = glob.glob("/kaggle/input/**/train.bin", recursive=True)
if not cands:
    raise SystemExit("train.bin not found. Add Input -> uzbek-fineweb2-tokens-16k")
TRAIN_BIN = cands[0]
VAL_BIN   = os.path.join(os.path.dirname(TRAIN_BIN), "val.bin")
train_ids = np.memmap(TRAIN_BIN, dtype=np.uint16, mode="r")
val_ids   = np.memmap(VAL_BIN,   dtype=np.uint16, mode="r")
print(f"train.bin {len(train_ids):,} tokens | val.bin {len(val_ids):,} tokens")

IDX_FILE = f"{WORK}/train_fingerprints.npy"
if os.path.exists(IDX_FILE):
    IDX = np.load(IDX_FILE); print(f"reusing index: {len(IDX):,} fingerprints")
else:
    t0 = time.time()
    IDX = np.unique(kgram_hashes(np.asarray(train_ids)))
    np.save(IDX_FILE, IDX)
    print(f"built {len(IDX):,} fingerprints in {time.time()-t0:.0f}s ({IDX.nbytes/1e6:.0f} MB)")

def dup_fraction(tok_ids):
    h = kgram_hashes(np.asarray(tok_ids, dtype=np.uint16))
    if len(h) == 0: return 0.0, 0
    p = np.searchsorted(IDX, h); p[p >= len(IDX)] = 0
    return float((IDX[p] == h).mean()), len(h)

# ============ STAGE 2 — deduplicated eval set from val.bin ==================
stage(2, "build the eval set from val documents that are absent from train.bin")
tk = AutoTokenizer.from_pretrained(MY_TOK_REPO); tk.model_max_length = 10**9
CLEAN_FILE = f"{WORK}/uz_heldout_dedup.txt"

if os.path.exists(CLEAN_FILE):
    TEXT = open(CLEAN_FILE, encoding="utf-8").read()
    print(f"reusing {CLEAN_FILE}")
    R["docs_kept"] = R.get("docs_kept"); R["docs_rejected"] = R.get("docs_rejected")
else:
    kept, rejected, words, cur, i, shown = [], 0, 0, [], 0, 0
    while i < len(val_ids) and val_ids[i] != EOT_ID:   # start after a boundary
        i += 1
    i += 1
    while i < len(val_ids) and words < EVAL_WORDS:
        t = int(val_ids[i])
        if t == EOT_ID:
            if len(cur) >= MIN_DOC_WORDS:
                frac, n_fp = dup_fraction(cur)
                if n_fp > 0 and frac <= MAX_DUP_FRAC:
                    d = tk.decode(cur).strip()
                    if d:
                        kept.append(d); words += len(d.split())
                else:
                    rejected += 1
                    if shown < 3 and n_fp > 0 and frac > 0:
                        snip = tk.decode(cur[:40]).strip().replace("\n", " ")
                        print(f"  rejected ({frac:.0%} of windows in train): {snip[:110]}...")
                        shown += 1
            cur = []
        else:
            cur.append(t)
        i += 1
    TEXT = "\n".join(kept)
    open(CLEAN_FILE, "w", encoding="utf-8").write(TEXT)
    total = len(kept) + rejected
    print(f"\nkept {len(kept):,} documents, rejected {rejected:,} "
          f"({rejected/max(total,1):.1%} of val documents are duplicated in train)")
    R["docs_kept"], R["docs_rejected"] = len(kept), rejected

EVAL_BYTES = len(TEXT.encode("utf-8"))
print(f"eval set: {len(TEXT.split()):,} words, {EVAL_BYTES:,} bytes")
R["clean_eval_bytes"], R["clean_eval_words"] = EVAL_BYTES, len(TEXT.split())

# ==================== STAGE 3 — verify with exact search ====================
stage(3, "verify: exact search for eval passages inside train.bin")
eval_tok = np.asarray(tk(TEXT, add_special_tokens=False)["input_ids"], dtype=np.uint16)
pos = np.linspace(0, len(eval_tok)-PROBE_TOK-1, N_PROBES).astype(int)
hits = 0
for n, p in enumerate(pos, 1):
    h = find_sequence(train_ids, np.asarray(eval_tok[p:p+PROBE_TOK], dtype=np.uint16))
    hits += bool(h)
    print(f"  probe {n:>2}  eval tok {p:>9,}  " + (f"FOUND @{h[0]:,}" if h else "absent"))
print(f"\n{hits}/{N_PROBES} probes found in train.bin")
R["probes_in_train"] = f"{hits}/{N_PROBES}"
if hits > 0:
    raise SystemExit(f"ABORT — {hits}/{N_PROBES} still leaking. Lower KEEP_MASK to 15 "
                     f"(denser index) and rerun.")
print("eval set verified clean — proceeding")

# ============ STAGE 4 — the old contaminated set, for the delta =============
stage(4, "rebuild the OLD contaminated eval set (for the memorisation delta)")
OLD_FILE = f"{WORK}/uz_heldout.txt"
if os.path.exists(OLD_FILE):
    OLD_TEXT = open(OLD_FILE, encoding="utf-8").read(); print(f"reusing {OLD_FILE}")
else:
    ds = load_dataset("HuggingFaceFW/fineweb-2", name="uzn_Latn", split="train", streaming=True)
    docs, w = [], 0
    for ex in ds:
        t = (ex.get("text") or "").strip()
        if not t: continue
        docs.append(t); w += len(t.split())
        if w >= EVAL_WORDS: break
    OLD_TEXT = "\n".join(docs); open(OLD_FILE, "w", encoding="utf-8").write(OLD_TEXT)
print(f"old eval set: {len(OLD_TEXT.split()):,} words")

# ================== STAGE 5 — adaptation pool + leak check ==================
stage(5, "adaptation pool")
POOL_FILE = f"{WORK}/adapt_dedup_{ADAPT_TOKENS}.npy"
tok_m = AutoTokenizer.from_pretrained(BASE)
if tok_m.pad_token is None: tok_m.pad_token = tok_m.eos_token
tok_m.model_max_length = 10**9

if os.path.exists(POOL_FILE):
    POOL = np.load(POOL_FILE); pool_text = ""
    print(f"reusing pool: {len(POOL)/1e6:.2f}M tokens")
else:
    need = int(ADAPT_TOKENS * 1.10)
    ds = load_dataset("HuggingFaceFW/fineweb-2", name="uzn_Latn", split="train", streaming=True)
    ids, chunks = [], []
    for ex in ds:
        t = (ex.get("text") or "").strip()
        if not t: continue
        chunks.append(t)
        ids.extend(tok_m(t, add_special_tokens=False)["input_ids"] + [tok_m.eos_token_id])
        if len(ids) >= need: break
    POOL = np.asarray(ids[:need], dtype=np.int32); np.save(POOL_FILE, POOL)
    pool_text = "\n".join(chunks)
    print(f"tokenized {len(POOL)/1e6:.2f}M tokens from {len(chunks):,} docs")

if pool_text:
    probes = [TEXT[i:i+300] for i in np.linspace(0, max(len(TEXT)-400, 1), 12).astype(int)]
    leak = sum(1 for p in probes if p in pool_text)
    print(f"eval-text probes inside the adaptation pool: {leak}/{len(probes)}")
    R["adapt_pool_leak"] = f"{leak}/{len(probes)}"

# ======================= STAGE 6 — QLoRA fine-tune ==========================
stage(6, f"QLoRA mGPT-1.3B (ctx {CTX_TRAIN}, all-linear r=16, ~{ADAPT_TOKENS/1e6:.0f}M tokens)")
if os.path.exists(f"{ADAPTER_DIR}/adapter_model.safetensors"):
    print(f"reusing adapter: {ADAPTER_DIR}")
else:
    class Blocks(Dataset):
        def __init__(s, a, c): s.a, s.c, s.n = a, c, len(a)//c
        def __len__(s): return s.n
        def __getitem__(s, i):
            x = s.a[i*s.c:(i+1)*s.c].astype(np.int64)
            return {"input_ids": torch.from_numpy(x), "labels": torch.from_numpy(x.copy())}
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_compute_dtype=DT, bnb_4bit_use_double_quant=True)
    m = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb, device_map={"": 0})
    m.config.use_cache = False
    m = prepare_model_for_kbit_training(m, use_gradient_checkpointing=True)
    m = get_peft_model(m, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
                                     task_type="CAUSAL_LM", target_modules="all-linear"))
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f"trainable: {trainable:,}"); R["trainable_params"] = trainable
    steps = ADAPT_TOKENS // (PER_DEV * ACC * CTX_TRAIN)
    print(f"steps: {steps} x {PER_DEV*ACC*CTX_TRAIN} = {steps*PER_DEV*ACC*CTX_TRAIN:,} tokens")
    args = TrainingArguments(
        output_dir=ADAPTER_DIR, per_device_train_batch_size=PER_DEV,
        gradient_accumulation_steps=ACC, max_steps=steps, learning_rate=LR,
        lr_scheduler_type="cosine", warmup_steps=max(10, steps//20),
        logging_steps=50, save_strategy="no", report_to=[],
        fp16=not BF16, bf16=BF16, gradient_checkpointing=True,
        optim="paged_adamw_8bit", seed=SEED)
    t0 = time.time()
    Trainer(model=m, args=args, train_dataset=Blocks(POOL, CTX_TRAIN),
            data_collator=default_data_collator).train()
    print(f"trained in {(time.time()-t0)/60:.1f} min")
    m.save_pretrained(ADAPTER_DIR); del m; free()

# ============================ model loaders =================================
def load_from_scratch():
    mp = hf_hub_download(MY_MODEL_REPO, "model.py"); sys.path.insert(0, os.path.dirname(mp))
    from model import GPT
    g = GPT(vocab_size=16384, n_embd=768, block_size=1024, num_heads=12, n_layers=12)
    g.load_state_dict(load_file(hf_hub_download(MY_MODEL_REPO, "model.safetensors")), strict=False)
    return g.eval().to(DEV)

def load_mgpt(adapter=None):
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_compute_dtype=DT, bnb_4bit_use_double_quant=True)
    mm = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb,
                                              device_map={"": 0}).eval()
    return PeftModel.from_pretrained(mm, adapter).eval() if adapter else mm

def logits_of(model, x):
    out = model(x)
    return out[0] if isinstance(out, tuple) else getattr(out, "logits", out)

# ========================= STAGE 7 — Protocol A =============================
stage(7, "Protocol A — 512-token chunks")

@torch.no_grad()
def protocol_a(tok, model, text, tag):
    ids = tok(text, add_special_tokens=False)["input_ids"]
    n = len(ids) // CHUNK_TOKENS
    C = torch.tensor(ids[:n*CHUNK_TOKENS], dtype=torch.long).view(n, CHUNK_TOKENS)
    nats, byts, ntok = 0.0, 0, 0
    for i in range(n):
        x = C[i:i+1].to(DEV)
        lg = logits_of(model, x); sl, st = lg[:, :-1, :].float(), x[:, 1:]
        nats += torch.nn.functional.cross_entropy(
            sl.reshape(-1, sl.size(-1)), st.reshape(-1), reduction="sum").item()
        byts += len(tok.decode(st[0].tolist()).encode("utf-8")); ntok += st.numel()
    out = {"bpb": nats/(LN2*byts), "ce": nats/ntok, "chunks": n, "bytes": byts}
    print(f"  [{tag}] bpb={out['bpb']:.4f}  CE={out['ce']:.4f}  chunks={n}")
    return out

# ========================= STAGE 8 — Protocol B =============================
def build_spans(text, span_bytes):
    spans, i, n, k = [], 0, len(text), 0
    while i < n:
        j, b = i, 0
        while j < n and b < span_bytes:
            b += len(text[j].encode("utf-8")); j += 1
        spans.append((f"span_{k:05d}", i, j)); i, k = j, k+1
    if spans and len(text[spans[-1][1]:spans[-1][2]].encode("utf-8")) < span_bytes//2:
        spans.pop()
    return spans

SPANS = build_spans(TEXT, SPAN_BYTES)

@torch.no_grad()
def protocol_b(tok, model, tag):
    rows, over = [], 0
    for span_id, a, b in SPANS:
        n_bytes = len(TEXT[a:b].encode("utf-8"))
        c, got = a, 0
        while c > 0 and got < CONTEXT_BYTES:
            c -= 1; got += len(TEXT[c].encode("utf-8"))
        enc = tok(TEXT[c:b], return_offsets_mapping=True, add_special_tokens=False)
        ids, offs = enc["input_ids"], enc["offset_mapping"]
        first = next((k for k, (s, e) in enumerate(offs) if s >= a - c), len(ids))
        if len(ids) < 2: continue
        if len(ids) > MAX_TOKENS:
            over += 1; drop = len(ids) - MAX_TOKENS
            ids = ids[drop:]; first = max(0, first - drop)
        x = torch.tensor(ids, dtype=torch.long, device=DEV).unsqueeze(0)
        lg = logits_of(model, x)
        logits = lg[:, :-1, :].float(); target = x[:, 1:].clone()
        if first > 1: target[:, :first-1] = -100
        nats = torch.nn.functional.cross_entropy(
            logits.reshape(-1, logits.size(-1)), target.reshape(-1),
            ignore_index=-100, reduction="sum").item()
        rows.append((span_id, nats, n_bytes))
    path = f"{WORK}/dedup_chunks_{tag}.csv"
    with open(path, "w", newline="", encoding="utf-8") as fh:
        w = csv.writer(fh); w.writerow(["chunk_id","sum_nats","n_bytes"])
        w.writerows([(i, f"{v:.6f}", n) for i, v, n in rows])
    bpb = sum(r[1] for r in rows)/(LN2*sum(r[2] for r in rows))
    print(f"  [{tag}] bpb={bpb:.4f}  spans={len(rows)}" + (f"  ({over} truncated)" if over else ""))
    return bpb

A, B = {}, {}
g = load_from_scratch()
A["from_scratch_clean"] = protocol_a(tk, g, TEXT, "from_scratch / CLEAN")
A["from_scratch_old"]   = protocol_a(tk, g, OLD_TEXT, "from_scratch / OLD (contaminated)")
stage(8, f"Protocol B — {len(SPANS)} byte-aligned spans")
B["from_scratch"] = protocol_b(tk, g, "from_scratch")
del g; free()

base = load_mgpt()
A["mgpt_base"] = protocol_a(tok_m, base, TEXT, "mgpt_base / CLEAN")
B["mgpt_base"] = protocol_b(tok_m, base, "mgpt_base")
del base; free()

q = load_mgpt(adapter=ADAPTER_DIR)
A["mgpt_qlora"] = protocol_a(tok_m, q, TEXT, "mgpt_qlora / CLEAN")
B["mgpt_qlora"] = protocol_b(tok_m, q, "mgpt_qlora")
del q; free()

R["protocol_a"], R["protocol_b"] = A, B
DELTA = A["from_scratch_clean"]["bpb"] - A["from_scratch_old"]["bpb"]
R["contamination_delta_bpb"] = DELTA

# ======================= STAGE 9 — paired bootstrap =========================
stage(9, f"paired bootstrap, {RESAMPLES:,} resamples")

def load_csv(tag):
    d = {}
    with open(f"{WORK}/dedup_chunks_{tag}.csv", encoding="utf-8") as fh:
        for r in csv.DictReader(fh):
            d[r["chunk_id"]] = (float(r["sum_nats"]), float(r["n_bytes"]))
    return d

def paired(ta, tb):
    a, b = load_csv(ta), load_csv(tb)
    assert set(a) == set(b), "span sets differ"
    ids = sorted(a)
    na = np.array([a[i][0] for i in ids]); ba = np.array([a[i][1] for i in ids])
    nb = np.array([b[i][0] for i in ids]); bb = np.array([b[i][1] for i in ids])
    assert np.allclose(ba, bb), "paired spans differ in bytes"
    rng = np.random.default_rng(SEED)
    idx = rng.integers(0, len(ids), size=(RESAMPLES, len(ids)))
    d = (nb[idx].sum(1)/(LN2*bb[idx].sum(1))) - (na[idx].sum(1)/(LN2*ba[idx].sum(1)))
    lo, hi = np.percentile(d, [2.5, 97.5])
    return {"diff": nb.sum()/(LN2*bb.sum()) - na.sum()/(LN2*ba.sum()),
            "ci_lo": lo, "ci_hi": hi, "excludes_zero": bool(lo > 0 or hi < 0),
            "p_le_zero": float((d <= 0).mean()), "n_spans": len(ids)}

boots = {"vs_qlora": paired("from_scratch", "mgpt_qlora"),
         "vs_base":  paired("from_scratch", "mgpt_base")}
R["bootstrap"] = boots

# ============================ STAGE 10 — results ============================
stage(10, "RESULTS")
bq, bb_ = boots["vs_qlora"], boots["vs_base"]
print(f"""
eval set            val.bin documents with zero 24-token overlap with train.bin
                    {R['clean_eval_words']:,} words, {EVAL_BYTES:,} bytes
documents           kept {R.get('docs_kept','?')}, rejected {R.get('docs_rejected','?')} as duplicates
exact-search probes {R['probes_in_train']}  (0/{N_PROBES} required)
adaptation          {ADAPT_TOKENS:,} tokens, ctx {CTX_TRAIN}, all-linear r=16

MEMORISATION EFFECT (same model, same protocol, two eval sets)
  from-scratch on OLD   (trained on it)   {A['from_scratch_old']['bpb']:.4f} bpb
  from-scratch on CLEAN (never seen)      {A['from_scratch_clean']['bpb']:.4f} bpb
  memorisation was worth                  {DELTA:+.4f} bpb

PROTOCOL A — 512-token chunks, CLEAN eval set
  model                  bpb      per-token CE
  from-scratch 103M    {A['from_scratch_clean']['bpb']:.4f}      {A['from_scratch_clean']['ce']:.4f}
  mGPT-1.3B base       {A['mgpt_base']['bpb']:.4f}      {A['mgpt_base']['ce']:.4f}
  mGPT-1.3B + QLoRA    {A['mgpt_qlora']['bpb']:.4f}      {A['mgpt_qlora']['ce']:.4f}
  paper claimed        1.1050 / 1.1628 / 1.1214  (on contaminated text)

PROTOCOL B — {len(SPANS)} byte-aligned spans, CLEAN eval set
  from-scratch 103M    {B['from_scratch']:.4f}
  mGPT-1.3B base       {B['mgpt_base']:.4f}
  mGPT-1.3B + QLoRA    {B['mgpt_qlora']:.4f}

PAIRED BOOTSTRAP ({RESAMPLES:,} resamples, seed {SEED}; positive = from-scratch better)
  vs QLoRA baseline  {bq['diff']:+.4f} bpb  95% CI [{bq['ci_lo']:+.4f}, {bq['ci_hi']:+.4f}]  excludes zero: {'YES' if bq['excludes_zero'] else 'NO'}  P(<=0)={bq['p_le_zero']:.4f}
  vs zero-shot base  {bb_['diff']:+.4f} bpb  95% CI [{bb_['ci_lo']:+.4f}, {bb_['ci_hi']:+.4f}]  excludes zero: {'YES' if bb_['excludes_zero'] else 'NO'}  P(<=0)={bb_['p_le_zero']:.4f}
""")

json.dump(R, open(f"{WORK}/results_v3.json", "w"), indent=2, default=float)
print(f"saved -> {WORK}/results_v3.json, dedup_chunks_*.csv, {CLEAN_FILE}")


GPU: Tesla T4 | bf16: False

STAGE 1 — fingerprint every 24-token window of train.bin
train.bin 954,981,404 tokens | val.bin 106,109,045 tokens
built 27,405,976 fingerprints in 141s (219 MB)

STAGE 2 — build the eval set from val documents that are absent from train.bin
  rejected (71% of windows in train): 31 iyul 2011 Xalqaro kuzatuvchilarga ko‘ra, ikki yil oldin Urumchida yuz bergan zo‘ravonliklardan beri Xitoy u...
  rejected (6% of windows in train): - Подробности - Опубликовано: 18.07.2015 01:05 Insoniyatni doimo bir muammo o‘ylantirgani-o‘y...
  rejected (100% of windows in train): Bu mukofot har yili “Eng yaxshi badiiy film”, “Eng yaxshi televizionniy film”, “Eng yaxshi bolalar va o‘smirla...

kept 474 documents, rejected 306 (39.2% of val documents are duplicated in train)
eval set: 200,818 words, 1,646,167 bytes

STAGE 3 — verify: exact search for eval passages inside train.bin
  probe  1  eval tok         0  absent
  probe  2  eval tok    23,443  absent
  probe  3  eval tok 

**Reading the "memorisation effect" line.** It is not a memorisation measurement. The two held-out sets are different text, and mGPT, which trained on neither, also scores much better on the clean set. Step 3 uses mGPT as a control and explains the difference.

## Step 3 — The tokenizer ablation

The ablation model has the same architecture as uzbek-gpt-103m and was trained on the same text for the same number of steps, with one change: mGPT's 100k tokenizer instead of the 16k Uzbek one. The larger vocabulary makes it 232M parameters.

Because data and training are held constant, the gap between it and uzbek-gpt-103m isolates the tokenizer's contribution.

Part 1 of this step also explains the "memorisation" line from Step 2: the old held-out set has 0.51% Cyrillic characters against 0.20% in the clean one, and Cyrillic is expensive for a Latin-only tokenizer but cheap for multilingual mGPT.

In [ ]:
"""
================================================================================
ABLATION RE-SCORE  +  EVAL-SET DIAGNOSTIC
================================================================================
Run in the SAME Kaggle session as 02_clean_eval_and_score.py (it reuses the files it
wrote to /kaggle/working). Add your ablation_best.pt dataset as an Input first.

Two jobs:
  PART 1  diagnose WHY the clean eval set scores so differently from the old one
          — no GPU, ~5 seconds. The -0.081 "memorisation" number is confounded
          and this shows by how much.
  PART 2  score the 232M tokenizer-ablation model on the clean eval set, under
          both protocols, and bootstrap it against the from-scratch model.

~10 minutes. No retraining.
================================================================================
"""

import os, sys, glob, math, csv, json
import numpy as np
import torch

LN2  = math.log(2.0)
WORK = "/kaggle/working"
CLEAN_FILE = f"{WORK}/uz_heldout_dedup.txt"
OLD_FILE   = f"{WORK}/uz_heldout.txt"
for f in (CLEAN_FILE, OLD_FILE, f"{WORK}/dedup_chunks_from_scratch.csv"):
    if not os.path.exists(f):
        raise SystemExit(f"missing {f} — run 02_clean_eval_and_score.py in this session first")

TEXT     = open(CLEAN_FILE, encoding="utf-8").read()
OLD_TEXT = open(OLD_FILE,   encoding="utf-8").read()

CHUNK_TOKENS, SPAN_BYTES, CONTEXT_BYTES, MAX_TOKENS = 512, 800, 300, 512
RESAMPLES, SEED = 10_000, 20260913

# ============================== PART 1 ======================================
print("="*72); print("PART 1 — why do the two eval sets score so differently?"); print("="*72)

def profile(t, name):
    n = len(t)
    cyr  = sum(1 for c in t if '\u0400' <= c <= '\u04FF')
    lat  = sum(1 for c in t if ('a' <= c.lower() <= 'z') or c in "oʻgʻʼ‘’")
    dig  = sum(1 for c in t if c.isdigit())
    punc = sum(1 for c in t if not c.isalnum() and not c.isspace())
    words = t.split()
    print(f"  {name:<10} {n:>10,} chars | Cyrillic {cyr/n:6.2%} | Latin {lat/n:6.2%} "
          f"| digits {dig/n:5.2%} | punct {punc/n:5.2%} | mean word {np.mean([len(w) for w in words]):.2f}")
    return {"chars": n, "cyrillic_frac": cyr/n, "digit_frac": dig/n}

p_old   = profile(OLD_TEXT, "OLD")
p_clean = profile(TEXT,     "CLEAN")

print(f"""
  The from-scratch model's tokenizer was trained on Uzbek LATIN only. Cyrillic
  falls back to bytes, which is expensive for it and cheap for multilingual mGPT.
  If OLD carries more Cyrillic than CLEAN, that alone moves the two models by
  different amounts and the -0.081 'memorisation' figure is not memorisation.
""")

# mGPT as a control: it never trained on either set, so its shift between the two
# eval sets is pure text-difficulty. Subtract it to isolate anything model-specific.
FS_OLD, FS_CLEAN = 1.1078, 1.0264          # from v3 Protocol A
MG_OLD, MG_CLEAN = 1.1653, 1.1091          # 1.1653 = old npz, chunk-byte denominator
print(f"  from-scratch   OLD {FS_OLD:.4f} -> CLEAN {FS_CLEAN:.4f}   delta {FS_CLEAN-FS_OLD:+.4f}")
print(f"  mGPT base      OLD {MG_OLD:.4f} -> CLEAN {MG_CLEAN:.4f}   delta {MG_CLEAN-MG_OLD:+.4f}  <- control")
print(f"  difference-in-differences                  {(FS_CLEAN-FS_OLD)-(MG_CLEAN-MG_OLD):+.4f}")
print("  A NEGATIVE diff-in-diff means the from-scratch model did RELATIVELY BETTER")
print("  on text it never saw — the opposite of a memorisation benefit.\n")

# ============================== PART 2 ======================================
print("="*72); print("PART 2 — score the 232M tokenizer ablation on the clean set"); print("="*72)

from transformers import AutoTokenizer
from huggingface_hub import hf_hub_download

ck_paths = glob.glob("/kaggle/input/**/ablation_best.pt", recursive=True) or \
           glob.glob("/kaggle/input/**/ablation_last.pt", recursive=True)
if not ck_paths:
    raise SystemExit("ablation checkpoint not found. Add Input -> your ablation dataset.")
CKPT = ck_paths[0]
print(f"checkpoint: {CKPT} ({os.path.getsize(CKPT)/1e6:.0f} MB)")

mp = hf_hub_download("IslombekT/uzbek-gpt-103m", "model.py")
sys.path.insert(0, os.path.dirname(mp))
from model import GPT

ck  = torch.load(CKPT, map_location="cpu", weights_only=False)
cfg = ck["config"]
print(f"config from checkpoint: {cfg}")
print(f"saved at step {ck.get('step')} | val {ck.get('val')}")

abl = GPT(cfg["vocab_size"], cfg["n_embd"], cfg["block_size"], cfg["n_head"], cfg["n_layer"])
abl.load_state_dict(ck["model"]); abl = abl.eval().cuda()
n_par = sum(p.numel() for p in abl.parameters())
print(f"ablation model: {n_par/1e6:.0f}M parameters\n")

tok = AutoTokenizer.from_pretrained("ai-forever/mGPT")   # the ablation's tokenizer
tok.model_max_length = 10**9

def logits_of(model, x):
    out = model(x)
    return out[0] if isinstance(out, tuple) else getattr(out, "logits", out)

@torch.no_grad()
def protocol_a(text, tag):
    ids = tok(text, add_special_tokens=False)["input_ids"]
    n = len(ids)//CHUNK_TOKENS
    C = torch.tensor(ids[:n*CHUNK_TOKENS], dtype=torch.long).view(n, CHUNK_TOKENS)
    nats, byts, ntok = 0.0, 0, 0
    for i in range(n):
        x = C[i:i+1].cuda()
        lg = logits_of(abl, x); sl, st = lg[:, :-1, :].float(), x[:, 1:]
        nats += torch.nn.functional.cross_entropy(
            sl.reshape(-1, sl.size(-1)), st.reshape(-1), reduction="sum").item()
        byts += len(tok.decode(st[0].tolist()).encode("utf-8")); ntok += st.numel()
    print(f"  [{tag}] bpb={nats/(LN2*byts):.4f}  CE={nats/ntok:.4f}  chunks={n}")
    return nats/(LN2*byts)

def build_spans(text, span_bytes):
    spans, i, n, k = [], 0, len(text), 0
    while i < n:
        j, b = i, 0
        while j < n and b < span_bytes:
            b += len(text[j].encode("utf-8")); j += 1
        spans.append((f"span_{k:05d}", i, j)); i, k = j, k+1
    if spans and len(text[spans[-1][1]:spans[-1][2]].encode("utf-8")) < span_bytes//2:
        spans.pop()
    return spans

SPANS = build_spans(TEXT, SPAN_BYTES)

@torch.no_grad()
def protocol_b(tag):
    rows, over = [], 0
    for span_id, a, b in SPANS:
        n_bytes = len(TEXT[a:b].encode("utf-8"))
        c, got = a, 0
        while c > 0 and got < CONTEXT_BYTES:
            c -= 1; got += len(TEXT[c].encode("utf-8"))
        enc = tok(TEXT[c:b], return_offsets_mapping=True, add_special_tokens=False)
        ids, offs = enc["input_ids"], enc["offset_mapping"]
        first = next((k for k, (s, e) in enumerate(offs) if s >= a-c), len(ids))
        if len(ids) < 2: continue
        if len(ids) > MAX_TOKENS:
            over += 1; drop = len(ids)-MAX_TOKENS
            ids = ids[drop:]; first = max(0, first-drop)
        x = torch.tensor(ids, dtype=torch.long, device="cuda").unsqueeze(0)
        lg = logits_of(abl, x)
        logits = lg[:, :-1, :].float(); target = x[:, 1:].clone()
        if first > 1: target[:, :first-1] = -100
        rows.append((span_id, torch.nn.functional.cross_entropy(
            logits.reshape(-1, logits.size(-1)), target.reshape(-1),
            ignore_index=-100, reduction="sum").item(), n_bytes))
    path = f"{WORK}/dedup_chunks_ablation.csv"
    with open(path, "w", newline="", encoding="utf-8") as fh:
        w = csv.writer(fh); w.writerow(["chunk_id","sum_nats","n_bytes"])
        w.writerows([(i, f"{v:.6f}", n) for i, v, n in rows])
    bpb = sum(r[1] for r in rows)/(LN2*sum(r[2] for r in rows))
    print(f"  [{tag}] bpb={bpb:.4f}  spans={len(rows)}" + (f"  ({over} truncated)" if over else ""))
    return bpb

a_clean = protocol_a(TEXT,     "ablation / CLEAN")
a_old   = protocol_a(OLD_TEXT, "ablation / OLD (contaminated)")
b_clean = protocol_b("ablation / spans")

# --------------------------- paired bootstrap -------------------------------
def load_csv(tag):
    d = {}
    with open(f"{WORK}/dedup_chunks_{tag}.csv", encoding="utf-8") as fh:
        for r in csv.DictReader(fh):
            d[r["chunk_id"]] = (float(r["sum_nats"]), float(r["n_bytes"]))
    return d

def paired(ta, tb):
    a, b = load_csv(ta), load_csv(tb)
    assert set(a) == set(b), "span sets differ"
    ids = sorted(a)
    na = np.array([a[i][0] for i in ids]); ba = np.array([a[i][1] for i in ids])
    nb = np.array([b[i][0] for i in ids]); bb = np.array([b[i][1] for i in ids])
    rng = np.random.default_rng(SEED)
    idx = rng.integers(0, len(ids), size=(RESAMPLES, len(ids)))
    d = (nb[idx].sum(1)/(LN2*bb[idx].sum(1))) - (na[idx].sum(1)/(LN2*ba[idx].sum(1)))
    lo, hi = np.percentile(d, [2.5, 97.5])
    return {"diff": nb.sum()/(LN2*bb.sum()) - na.sum()/(LN2*ba.sum()),
            "ci_lo": lo, "ci_hi": hi, "excludes_zero": bool(lo > 0 or hi < 0),
            "p_le_zero": float((d <= 0).mean())}

bt = paired("from_scratch", "ablation")

print(f"""
{'='*72}
ABLATION RESULTS
{'='*72}
ablation model      {n_par/1e6:.0f}M params, mGPT tokenizer (vocab {cfg['vocab_size']:,})
                    trained on the same corpus text, same architecture

  ablation on OLD   (contaminated)   {a_old:.4f} bpb
  ablation on CLEAN (never seen)     {a_clean:.4f} bpb
  paper claimed                      1.1579 bpb (on contaminated text)

PROTOCOL A — 512-token chunks, CLEAN
  from-scratch 103M, own tokenizer   1.0264
  ablation     {n_par/1e6:.0f}M, mGPT tokenizer   {a_clean:.4f}

PROTOCOL B — byte-aligned spans, CLEAN  (the fair-context comparison)
  from-scratch 103M, own tokenizer   1.0281
  ablation     {n_par/1e6:.0f}M, mGPT tokenizer   {b_clean:.4f}

PAIRED BOOTSTRAP, from-scratch vs ablation (positive = dedicated tokenizer better)
  {bt['diff']:+.4f} bpb   95% CI [{bt['ci_lo']:+.4f}, {bt['ci_hi']:+.4f}]   excludes zero: {'YES' if bt['excludes_zero'] else 'NO'}   P(<=0)={bt['p_le_zero']:.4f}
{'='*72}
""")

json.dump({"eval_profile_old": p_old, "eval_profile_clean": p_clean,
           "ablation_params": int(n_par), "ablation_config": cfg,
           "ablation_a_clean": a_clean, "ablation_a_old": a_old,
           "ablation_b_clean": b_clean, "bootstrap_vs_from_scratch": bt},
          open(f"{WORK}/ablation_results.json", "w"), indent=2, default=float)
print(f"saved -> {WORK}/ablation_results.json")


PART 1 — why do the two eval sets score so differently?
  OLD         1,645,017 chars | Cyrillic  0.51% | Latin 82.51% | digits 0.86% | punct 4.35% | mean word 7.09
  CLEAN       1,609,252 chars | Cyrillic  0.20% | Latin 83.89% | digits 0.77% | punct 2.91% | mean word 7.01

  The from-scratch model's tokenizer was trained on Uzbek LATIN only. Cyrillic
  falls back to bytes, which is expensive for it and cheap for multilingual mGPT.
  If OLD carries more Cyrillic than CLEAN, that alone moves the two models by
  different amounts and the -0.081 'memorisation' figure is not memorisation.

  from-scratch   OLD 1.1078 -> CLEAN 1.0264   delta -0.0814
  mGPT base      OLD 1.1653 -> CLEAN 1.1091   delta -0.0562  <- control
  difference-in-differences                  -0.0252
  A NEGATIVE diff-in-diff means the from-scratch model did RELATIVELY BETTER
  on text it never saw — the opposite of a memorisation benefit.

PART 2 — score the 232M tokenizer ablation on the clean set
checkpoint: /kaggle

## Results (Protocol B, deduplicated held-out set)

| Model | Bits/byte | Gap to from-scratch [95% CI] |
|---|---|---|
| uzbek-gpt-103m (from scratch) | **1.0281** | — |
| Ablation, mGPT tokenizer (232M) | 1.0492 | +0.0211 [0.0195, 0.0227] |
| mGPT-1.3B + QLoRA, 1M tokens | 1.0766 (session 1: 1.0764) | +0.0485 [0.0447, 0.0522] (session 1: +0.0483 [0.0444, 0.0520]) |
| mGPT-1.3B zero-shot | 1.0774 | +0.0493 [0.0455, 0.0530] |

- **Tokenizer:** 0.021 bits/byte, measured at matched data.
- **In-domain pretraining:** the rest, about 0.027 against this 1M-token baseline.
- The budget sweep (0 / 1M / 10M adaptation tokens) is in part B.